<a href="https://colab.research.google.com/github/niksisons/neural_networks/blob/main/8.%20%D0%93%D0%B5%D0%BD%D0%B5%D1%80%D0%B0%D1%82%D0%B8%D0%B2%D0%BD%D0%BE-%D1%81%D0%BE%D1%81%D1%82%D1%8F%D0%B7%D0%B0%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F%20%D1%81%D0%B5%D1%82%D1%8C%20(GAN)/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%968_%D0%93%D0%B5%D0%BD%D0%B5%D1%80%D0%B0%D1%82%D0%B8%D0%B2%D0%BD%D0%BE_%D1%81%D0%BE%D1%81%D1%82%D1%8F%D0%B7%D0%B0%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F_%D1%81%D0%B5%D1%82%D1%8C_(GAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа №8. Генеративно-состязательная сеть (GAN)**

---
**❗ Примечание:**

Не забывайте периодически сохранять параметры модели. Функции для этого описаны в теоретической части. В случае приостановки процесса обучения из-за перегрузки ОЗУ, Вы сможете загрузить последние предобученные параметры и продолжить обучение.

---

## **Задание №1.** Обучите генератор воспризводить примитивные изображения. Датасет выберите по желанию. ([Пример №1](https://www.kaggle.com/datasets/andrewmvd/medical-mnist), [Пример №2](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/fashion_mnist/load_data#example), [Пример №3](https://www.kaggle.com/datasets/sagyamthapa/handwritten-math-symbols))





In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from IPython import display

# 1. Скачивание и подготовка датасета
!curl -L -o medical-mnist.zip https://www.kaggle.com/api/v1/datasets/download/andrewmvd/medical-mnist
!unzip -q -o medical-mnist.zip -d medical-mnist/

In [ ]:
# Параметры
BATCH_SIZE = 64
IMAGE_SIZE = (64, 64)
CHANNELS = 1


In [ ]:

# Загрузка датасета (градации серого)
dataset_path = "medical-mnist"
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    label_mode=None,
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Нормализация изображений в диапазон [-1, 1]
def normalize(image):
    image = tf.cast(image, tf.float32)
    image = (image - 127.5) / 127.5
    return image

train_dataset = train_dataset.map(normalize).prefetch(tf.data.AUTOTUNE)

# 2. Создание моделей
def make_generator_model():
    model = tf.keras.Sequential([
        layers.Dense(8*8*256, use_bias=False, input_shape=(100,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Reshape((8, 8, 256)),

        layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(), # 16x16x128

        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(), # 32x32x64

        layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh') # 64x64x1
    ])
    return model

def make_discriminator_model():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[64, 64, 1]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

generator = make_generator_model()
discriminator = make_discriminator_model()

# 3. Функция потерь и оптимизаторы
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

# 4. Сохранение контрольных точек (Checkpoint)
checkpoint_dir = './training_checkpoints/task1'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

EPOCHS = 50 
noise_dim = 100
num_examples_to_generate = 16

# Постоянный шум для визуализации прогресса
seed = tf.random.normal([num_examples_to_generate, noise_dim])

@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))
    return gen_loss, disc_loss

# 5. Цикл обучения
def train(dataset, epochs):
    gen_loss_history = []
    disc_loss_history = []
    
    for epoch in range(epochs):
        start = time.time()
        
        gen_loss_epoch = 0
        disc_loss_epoch = 0
        batches = 0
        
        for image_batch in dataset:
            g_loss, d_loss = train_step(image_batch)
            gen_loss_epoch += g_loss
            disc_loss_epoch += d_loss
            batches += 1
            
        gen_loss_history.append(gen_loss_epoch / batches)
        disc_loss_history.append(disc_loss_epoch / batches)

        # Сохранение модели каждые 5 эпох и в конце
        if (epoch + 1) % 5 == 0 or (epoch + 1) == epochs:
            checkpoint.save(file_prefix = checkpoint_prefix)

        print(f'Эпоха {epoch + 1} | Время: {time.time()-start:.2f} с | G-Loss: {gen_loss_history[-1]:.4f} | D-Loss: {disc_loss_history[-1]:.4f}')
        
    return gen_loss_history, disc_loss_history

# Запуск обучения
gen_hist, disc_hist = train(train_dataset, EPOCHS)

# Визуализация лоссов
plt.figure(figsize=(10, 5))
plt.plot(gen_hist, label='Loss Генератора')
plt.plot(disc_hist, label='Loss Дискриминатора')
plt.title('График потерь в процессе обучения базового GAN')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.legend()
plt.show()

### **Демонстрация сгенерированных изображений:**

In [ ]:
def generate_and_save_images(model, test_input):
    predictions = model(test_input, training=False)

    fig = plt.figure(figsize=(4, 4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        # Масштабируем пиксели из [-1, 1] обратно в [0, 1]
        img = predictions[i, :, :, 0] * 0.5 + 0.5
        plt.imshow(img, cmap='gray')
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

# Загружаем последнюю контрольную точку (для надежности)
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

# Генерируем изображения из фиксированного вектора шума
print("Сгенерированные изображения Базовой GAN:")
generate_and_save_images(generator, seed)

---

## **Задание №2.** Обучите генератор воспризводить примитивные изображения по заданному условию (Conditional Generative Adversarial Nets (CGAN)).



(На вход генератора подается вектор случайного шума и метка класса - на выходе должно получиться изображение, соответствующее данному классу)



Датасет выберите по желанию. ([Пример №1](https://www.kaggle.com/datasets/andrewmvd/medical-mnist), [Пример №2](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/fashion_mnist/load_data#example), [Пример №3](https://www.kaggle.com/datasets/sagyamthapa/handwritten-math-symbols))

In [ ]:
# 1. Загрузка датасета с метками классов
train_dataset_cgan = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    label_mode="int",
    color_mode="grayscale",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

class_names = train_dataset_cgan.class_names
num_classes = len(class_names)
print(f"Обнаруженные классы ({num_classes}):", class_names)

def normalize_cgan(image, label):
    image = tf.cast(image, tf.float32)
    image = (image - 127.5) / 127.5
    return image, label

train_dataset_cgan = train_dataset_cgan.map(normalize_cgan).prefetch(tf.data.AUTOTUNE)

# 2. Модели CGAN
# Условный Генератор
def build_cgan_generator():
    # Входы
    noise_input = layers.Input(shape=(noise_dim,))
    label_input = layers.Input(shape=(1,), dtype=tf.int32)
    
    # Эмбеддинг меток классов
    label_embedding = layers.Embedding(num_classes, noise_dim)(label_input)
    label_embedding = layers.Flatten()(label_embedding)
    
    # Объединение шума и метки (перемножение)
    model_input = layers.Multiply()([noise_input, label_embedding])
    
    # Основная сеть
    x = layers.Dense(8*8*256, use_bias=False)(model_input)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.Reshape((8, 8, 256))(x)
    
    x = layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x) # 16x16x128
    
    x = layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x) # 32x32x64
    
    out = layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh')(x) # 64x64x1
    
    return tf.keras.Model([noise_input, label_input], out)

# Условный Дискриминатор
def build_cgan_discriminator():
    # Входы
    image_input = layers.Input(shape=[64, 64, 1])
    label_input = layers.Input(shape=(1,), dtype=tf.int32)
    
    # Маска меток
    label_embedding = layers.Embedding(num_classes, 64*64)(label_input)
    label_embedding = layers.Reshape((64, 64, 1))(label_embedding)
    
    # Объединение картинки и маски
    concat = layers.Concatenate()([image_input, label_embedding])
    
    x = layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same')(concat)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Flatten()(x)
    out = layers.Dense(1)(x)
    
    return tf.keras.Model([image_input, label_input], out)

cgan_generator = build_cgan_generator()
cgan_discriminator = build_cgan_discriminator()

# 3. Оптимизаторы и Checkpoints
cgan_gen_optimizer = tf.keras.optimizers.Adam(1e-4)
cgan_disc_optimizer = tf.keras.optimizers.Adam(1e-4)

checkpoint_dir_cgan = './training_checkpoints/task2_cgan'
checkpoint_prefix_cgan = os.path.join(checkpoint_dir_cgan, "ckpt")
checkpoint_cgan = tf.train.Checkpoint(generator_optimizer=cgan_gen_optimizer,
                                 discriminator_optimizer=cgan_disc_optimizer,
                                 generator=cgan_generator,
                                 discriminator=cgan_discriminator)

# 4. Логика обучения CGAN
@tf.function
def train_step_cgan(images, labels):
    # Текущий размер батча
    batch_size = tf.shape(images)[0]
    noise = tf.random.normal([batch_size, noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = cgan_generator([noise, labels], training=True)

        real_output = cgan_discriminator([images, labels], training=True)
        fake_output = cgan_discriminator([generated_images, labels], training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    grad_gen = gen_tape.gradient(gen_loss, cgan_generator.trainable_variables)
    grad_disc = disc_tape.gradient(disc_loss, cgan_discriminator.trainable_variables)

    cgan_gen_optimizer.apply_gradients(zip(grad_gen, cgan_generator.trainable_variables))
    cgan_disc_optimizer.apply_gradients(zip(grad_disc, cgan_discriminator.trainable_variables))
    
    return gen_loss, disc_loss

def train_cgan(dataset, epochs):
    gen_loss_history = []
    disc_loss_history = []
    
    for epoch in range(epochs):
        start = time.time()
        gen_loss_epoch = 0; disc_loss_epoch = 0; batches = 0
        
        for image_batch, label_batch in dataset:
            g_loss, d_loss = train_step_cgan(image_batch, label_batch)
            gen_loss_epoch += g_loss
            disc_loss_epoch += d_loss
            batches += 1
            
        gen_loss_history.append(gen_loss_epoch / batches)
        disc_loss_history.append(disc_loss_epoch / batches)

        if (epoch + 1) % 5 == 0 or (epoch + 1) == epochs:
            checkpoint_cgan.save(file_prefix=checkpoint_prefix_cgan)

        print(f'CGAN Эпоха {epoch + 1} | Время: {time.time()-start:.2f} с | G-Loss: {gen_loss_history[-1]:.4f} | D-Loss: {disc_loss_history[-1]:.4f}')
        
    return gen_loss_history, disc_loss_history

CGAN_EPOCHS = 30
cgan_gen_hist, cgan_disc_hist = train_cgan(train_dataset_cgan, CGAN_EPOCHS)

# Визуализация лоссов CGAN
plt.figure(figsize=(10, 5))
plt.plot(cgan_gen_hist, label='Loss Генератора')
plt.plot(cgan_disc_hist, label='Loss Дискриминатора')
plt.title('График потерь в процессе обучения CGAN')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.legend()
plt.show()

### **Демонстрация сгенерированных изображений:**

In [ ]:
def generate_cgan_images(model, num_classes, class_names):
    # Генерируем по одной картинке для каждого класса
    num_examples = num_classes
    noise = tf.random.normal([num_examples, noise_dim])
    labels = tf.constant(range(num_classes))
    
    predictions = model([noise, labels], training=False)

    fig = plt.figure(figsize=(15, 6))

    for i in range(predictions.shape[0]):
        plt.subplot(1, num_classes, i+1)
        img = predictions[i, :, :, 0] * 0.5 + 0.5 # из [-1, 1] в [0, 1]
        plt.imshow(img, cmap='gray')
        plt.title(class_names[i])
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

# Загружаем последнюю контрольную точку (для надежности)
checkpoint_cgan.restore(tf.train.latest_checkpoint(checkpoint_dir_cgan))

print("CGAN: Предсказания генератора по каждому из заданных классов (condition)")
generate_cgan_images(cgan_generator, num_classes, class_names)

---

## **Задание №3.** Обучите генератор колоризировать изображения из выбранного Вами датасета (можете использовать датасет из работы №6, в которой Вы решали аналогичную задачу).

In [ ]:
# 1. Подготовка датасета для колоризации
# Для простоты и скорости загрузки здесь мы используем встроенный CIFAR-10, 
# который отлично подходит для базовой демонстрации колоризации, 
# но архитектура подойдет и для любого загруженного датасета.

(x_train_color, _), (x_test_color, _) = tf.keras.datasets.cifar10.load_data()

# Для ускорения возьмем подвыборку
x_train_color = x_train_color[:10000]

def process_images_colorization(image):
    image_color = tf.cast(image, tf.float32)
    image_color = (image_color / 127.5) - 1.0 # Нормализация [-1, 1]
    
    # Создаем черно-белую копию для входа
    image_gray = tf.image.rgb_to_grayscale(image_color)
    return image_gray, image_color

colorization_dataset = tf.data.Dataset.from_tensor_slices(x_train_color)
colorization_dataset = colorization_dataset.map(process_images_colorization)
colorization_dataset = colorization_dataset.batch(64).shuffle(1000).prefetch(tf.data.AUTOTUNE)

# 2. Архитектура U-Net для колоризации (Генератор)
def build_unet_generator():
    inputs = layers.Input(shape=[32, 32, 1])

    # Downsampling
    d1 = layers.Conv2D(64, (4, 4), strides=2, padding='same')(inputs)
    d1 = layers.LeakyReLU(0.2)(d1) # 16x16x64

    d2 = layers.Conv2D(128, (4, 4), strides=2, padding='same')(d1)
    d2 = layers.BatchNormalization()(d2)
    d2 = layers.LeakyReLU(0.2)(d2) # 8x8x128

    d3 = layers.Conv2D(256, (4, 4), strides=2, padding='same')(d2)
    d3 = layers.BatchNormalization()(d3)
    d3 = layers.LeakyReLU(0.2)(d3) # 4x4x256

    # Upsampling
    u1 = layers.Conv2DTranspose(128, (4, 4), strides=2, padding='same')(d3)
    u1 = layers.BatchNormalization()(u1)
    u1 = layers.Activation('relu')(u1)
    u1 = layers.Concatenate()([u1, d2]) # 8x8

    u2 = layers.Conv2DTranspose(64, (4, 4), strides=2, padding='same')(u1)
    u2 = layers.BatchNormalization()(u2)
    u2 = layers.Activation('relu')(u2)
    u2 = layers.Concatenate()([u2, d1]) # 16x16

    outputs = layers.Conv2DTranspose(3, (4, 4), strides=2, padding='same', activation='tanh')(u2) # 32x32x3

    return tf.keras.Model(inputs=inputs, outputs=outputs)

# Дискриминатор (PatchGAN)
def build_patchgan_discriminator():
    inp = layers.Input(shape=[32, 32, 1], name='input_image')
    tar = layers.Input(shape=[32, 32, 3], name='target_image')

    x = layers.Concatenate()([inp, tar]) # 32x32x4

    x = layers.Conv2D(64, (4, 4), strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x) # 16x16

    x = layers.Conv2D(128, (4, 4), strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x) # 8x8

    x = layers.Conv2D(256, (4, 4), strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x) # 4x4
    
    x = layers.Flatten()(x)
    outputs = layers.Dense(1)(x)

    return tf.keras.Model(inputs=[inp, tar], outputs=outputs)

generator_col = build_unet_generator()
discriminator_col = build_patchgan_discriminator()

# 3. Оптимизаторы и функции потерь
gen_opt_col = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
disc_opt_col = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
loss_obj = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss_col(disc_real_output, disc_generated_output):
    real_loss = loss_obj(tf.ones_like(disc_real_output), disc_real_output)
    generated_loss = loss_obj(tf.zeros_like(disc_generated_output), disc_generated_output)
    return real_loss + generated_loss

def generator_loss_col(disc_generated_output, gen_output, target):
    gan_loss = loss_obj(tf.ones_like(disc_generated_output), disc_generated_output)
    # L1 Loss (Средняя абсолютная ошибка)
    l1_loss = tf.reduce_mean(tf.abs(target - gen_output))
    total_gen_loss = gan_loss + (100 * l1_loss) # Коэффициент L1-лосса = 100
    return total_gen_loss

checkpoint_col_dir = './training_checkpoints/task3_col'
checkpoint_col = tf.train.Checkpoint(generator=generator_col, discriminator=discriminator_col,
                                     generator_optimizer=gen_opt_col, discriminator_optimizer=disc_opt_col)

# 4. Обучение
@tf.function
def train_step_col(input_image, target, epoch):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator_col(input_image, training=True)

        disc_real_output = discriminator_col([input_image, target], training=True)
        disc_generated_output = discriminator_col([input_image, gen_output], training=True)

        gen_total_loss = generator_loss_col(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss_col(disc_real_output, disc_generated_output)

    generator_gradients = gen_tape.gradient(gen_total_loss, generator_col.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss, discriminator_col.trainable_variables)

    gen_opt_col.apply_gradients(zip(generator_gradients, generator_col.trainable_variables))
    disc_opt_col.apply_gradients(zip(discriminator_gradients, discriminator_col.trainable_variables))
    return gen_total_loss, disc_loss

EPOCHS_COL = 30

def train_colorization(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()
        for input_image, target in dataset:
            g_loss, d_loss = train_step_col(input_image, target, epoch)
            
        print(f'Colorization Эпоха {epoch + 1} | Время: {time.time()-start:.2f} с')
        checkpoint_col.save(file_prefix=os.path.join(checkpoint_col_dir, "ckpt"))

print("Начинаем обучение GAN для колоризации...")
train_colorization(colorization_dataset, EPOCHS_COL)

### **Демонстрация сгенерированных изображений:**

In [ ]:
def generate_images_color(model, test_input, tar):
    prediction = model(test_input, training=True)
    
    plt.figure(figsize=(15, 5))
    display_list = [test_input[0], tar[0], prediction[0]]
    title = ['Original Grayscale', 'Ground Truth Color', 'Predicted Color']

    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.title(title[i])
        
        # Получаем данные
        img = display_list[i].numpy() * 0.5 + 0.5
        
        # Если черно-белое изображение (1 канал), то убираем последний дименшн
        if img.shape[-1] == 1:
            plt.imshow(img[:, :, 0], cmap='gray')
        else:
            plt.imshow(img)
            
        plt.axis('off')
        
    plt.show()

# Загружаем последнюю контрольную точку
checkpoint_col.restore(tf.train.latest_checkpoint(checkpoint_col_dir))

# Тест
for inp, tar in colorization_dataset.take(2): # 2 тестовых примера
    generate_images_color(generator_col, inp, tar)

---

## **Задание №4.** Обучите генератор воспроизводить изображения из выбранного Вами датасета (pix2pix).



---
Примеры таких датасетов представлены на сайтах [kaggle.com](https://www.kaggle.com/search?q=pix2pix+in%3Adatasets+datasetFileTypes%3Ajpg+datasetFileTypes%3Apng) и [efrosgans.eecs.berkeley.edu*](http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/).

**\*Калифорнийский университет Беркли (UC Berkeley) внесён в реестр организаций, деятельность которых признана нежелательной в России.**


---

In [ ]:
# 1. Загрузка Pix2Pix (Map to Satellite Dataset)
!curl -L -o pix2pix-maps.zip https://www.kaggle.com/api/v1/datasets/download/alincijov/pix2pix-maps
!unzip -q -o pix2pix-maps.zip -d pix2pix-maps/


In [ ]:

# Используем train папку (изображения склеены A и B: 600x1200 -> 600x600 * 2)
map_dataset_path = 'pix2pix-maps/train' 
BATCH_SIZE_P2P = 1
IMG_WIDTH = 256
IMG_HEIGHT = 256

def load_maps(image_file):
    image = tf.io.read_file(image_file)
    image = tf.image.decode_jpeg(image)
    
    w = tf.shape(image)[1]
    w = w // 2
    
    # Каждая картинка состоит из спутника и карты: [Map, Satellite]
    input_image = image[:, w:, :] # Спутник или Карта (зависит от того, что мы хотим предсказывать)
    real_image = image[:, :w, :]
    
    # Резайзим
    input_image = tf.cast(tf.image.resize(input_image, [IMG_HEIGHT, IMG_WIDTH]), tf.float32)
    real_image = tf.cast(tf.image.resize(real_image, [IMG_HEIGHT, IMG_WIDTH]), tf.float32)
    
    # Нормализуем к [-1, 1]
    input_image = (input_image / 127.5) - 1
    real_image = (real_image / 127.5) - 1

    return input_image, real_image

train_maps = tf.data.Dataset.list_files(os.path.join(map_dataset_path, '*.jpg'))
train_maps = train_maps.map(load_maps, num_parallel_calls=tf.data.AUTOTUNE)
train_maps = train_maps.shuffle(400).batch(BATCH_SIZE_P2P).prefetch(tf.data.AUTOTUNE)

# 2. Архитектура Pix2Pix (U-Net Generator + PatchGAN Discriminator)
# Поскольку архитектура Pix2Pix идентична задаче колоризации, 
# мы можем переиспользовать концепцию. Напишем её для размера 256x256x3.

# -- U-Net Encoder Блок --
def downsample(filters, size, apply_batchnorm=True):
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(layers.Conv2D(filters, size, strides=2, padding='same', kernel_initializer=initializer, use_bias=False))
    if apply_batchnorm: result.add(layers.BatchNormalization())
    result.add(layers.LeakyReLU())
    return result

# -- U-Net Decoder Блок --
def upsample(filters, size, apply_dropout=False):
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(layers.Conv2DTranspose(filters, size, strides=2, padding='same', kernel_initializer=initializer, use_bias=False))
    result.add(layers.BatchNormalization())
    if apply_dropout: result.add(layers.Dropout(0.5))
    result.add(layers.ReLU())
    return result

# 3. Полный Генератор для 256x256x3
def build_pix2pix_generator():
    inputs = layers.Input(shape=[256, 256, 3])
    
    down_stack = [
        downsample(64, 4, apply_batchnorm=False), # 128x128x64
        downsample(128, 4),                       # 64x64x128
        downsample(256, 4),                       # 32x32x256
        downsample(512, 4),                       # 16x16x512
        downsample(512, 4),                       # 8x8x512
        downsample(512, 4),                       # 4x4x512
        downsample(512, 4),                       # 2x2x512
        downsample(512, 4),                       # 1x1x512
    ]
    
    up_stack = [
        upsample(512, 4, apply_dropout=True), # 2x2x1024
        upsample(512, 4, apply_dropout=True), # 4x4x1024
        upsample(512, 4, apply_dropout=True), # 8x8x1024
        upsample(512, 4),                     # 16x16x1024
        upsample(256, 4),                     # 32x32x512
        upsample(128, 4),                     # 64x64x256
        upsample(64, 4),                      # 128x128x128
    ]

    initializer = tf.random_normal_initializer(0., 0.02)
    last = layers.Conv2DTranspose(3, 4, strides=2, padding='same', kernel_initializer=initializer, activation='tanh') # 256x256x3
    
    x = inputs
    skips = []
    
    for down in down_stack:
        x = down(x)
        skips.append(x)
        
    skips = reversed(skips[:-1])

    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = layers.Concatenate()([x, skip])
        
    x = last(x)

    return tf.keras.Model(inputs=inputs, outputs=x)

# 4. Полный Дискриминатор (PatchGAN)
def build_pix2pix_discriminator():
    inp = layers.Input(shape=[256, 256, 3], name='input_image')
    tar = layers.Input(shape=[256, 256, 3], name='target_image')
    x = layers.concatenate([inp, tar]) # 256x256x6
    
    down1 = downsample(64, 4, False)(x) # 128x128x64
    down2 = downsample(128, 4)(down1) # 64x64x128
    down3 = downsample(256, 4)(down2) # 32x32x256

    zero_pad1 = layers.ZeroPadding2D()(down3) # 34x34x256
    conv = layers.Conv2D(512, 4, strides=1, kernel_initializer=tf.random_normal_initializer(0., 0.02), use_bias=False)(zero_pad1) # 31x31x512
    batchnorm1 = layers.BatchNormalization()(conv)
    leaky_relu = layers.LeakyReLU()(batchnorm1)
    zero_pad2 = layers.ZeroPadding2D()(leaky_relu) # 33x33x512
    
    last = layers.Conv2D(1, 4, strides=1, kernel_initializer=tf.random_normal_initializer(0., 0.02))(zero_pad2) # 30x30x1

    return tf.keras.Model(inputs=[inp, tar], outputs=last)

generator_p2p = build_pix2pix_generator()
discriminator_p2p = build_pix2pix_discriminator()

# 5. Loss и Optimizers Прямо из туториала TensorFlow 
generator_optimizer_p2p = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer_p2p = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
checkpoint_p2p_dir = './training_checkpoints/task4_p2p'
checkpoint_p2p = tf.train.Checkpoint(generator_optimizer=generator_optimizer_p2p, discriminator_optimizer=discriminator_optimizer_p2p,
                                     generator=generator_p2p, discriminator=discriminator_p2p)

# Генератор Pix2Pix Loss (cGAN + L1 * 100)
LAMBDA = 100
bin_cross_obj = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss_p2p(disc_generated_output, gen_output, target):
    gan_loss = bin_cross_obj(tf.ones_like(disc_generated_output), disc_generated_output)
    l1_loss = tf.reduce_mean(tf.abs(target - gen_output))
    return gan_loss + (LAMBDA * l1_loss)

def discriminator_loss_p2p(disc_real_output, disc_generated_output):
    real_loss = bin_cross_obj(tf.ones_like(disc_real_output), disc_real_output)
    generated_loss = bin_cross_obj(tf.zeros_like(disc_generated_output), disc_generated_output)
    return real_loss + generated_loss

@tf.function
def train_step_p2p(input_image, target, epoch):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator_p2p(input_image, training=True)

        disc_real_output = discriminator_p2p([input_image, target], training=True)
        disc_generated_output = discriminator_p2p([input_image, gen_output], training=True)

        gen_total_loss = generator_loss_p2p(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss_p2p(disc_real_output, disc_generated_output)

    generator_gradients = gen_tape.gradient(gen_total_loss, generator_p2p.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss, discriminator_p2p.trainable_variables)

    generator_optimizer_p2p.apply_gradients(zip(generator_gradients, generator_p2p.trainable_variables))
    discriminator_optimizer_p2p.apply_gradients(zip(discriminator_gradients, discriminator_p2p.trainable_variables))

EPOCHS_P2P = 50 # Обучение может занять время. Поставьте 50-100 для качественных результатов
print("Начинаем обучение Pix2Pix (спутник-карта)...")

for epoch in range(EPOCHS_P2P):
    start = time.time()
    for n, (input_image, target) in train_maps.enumerate():
        train_step_p2p(input_image, target, epoch)
        if n % 100 == 0:
            print(".", end='') # Прогресс-бар

    print()
    print(f'Pix2Pix Эпоха {epoch + 1} завершена за {time.time()-start:.2f} сек.')
    checkpoint_p2p.save(file_prefix=os.path.join(checkpoint_p2p_dir, "ckpt"))

### **Демонстрация сгенерированных изображений:**

In [ ]:
def generate_images_p2p(model, test_input, tar):
    prediction = model(test_input, training=True)
    plt.figure(figsize=(15, 15))

    display_list = [test_input[0], tar[0], prediction[0]]
    title = ['Original Input', 'Ground Truth Output', 'Predicted Output']

    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.title(title[i])
        # getting the pixel values between [0, 1] to plot it.
        plt.imshow(display_list[i] * 0.5 + 0.5)
        plt.axis('off')
    plt.show()

# Загружаем последнюю контрольную точку
checkpoint_p2p.restore(tf.train.latest_checkpoint(checkpoint_p2p_dir))

# Демонстрация (Выводим 3 картинки)
print("Демонстрация результатов Pix2Pix модели:")
for inp, tar in train_maps.take(3):
    generate_images_p2p(generator_p2p, inp, tar)